### Fix find camera default resolutions

In [1]:
from lerobot.find_cameras import main, find_and_print_cameras, cleanup_cameras, find_all_opencv_cameras, save_images_from_all_cameras
from lerobot.cameras.opencv.camera_opencv import OpenCVCamera
import cv2
from pathlib import Path

# display(find_all_opencv_cameras())
# OpenCVCamera.find_cameras()

save_images_from_all_cameras(output_dir=Path("/home/doog/repository/lerobot/notebooks/output/"), record_time_s=3, camera_type="opencv")



Probing camera: /dev/video0
Camera /dev/video0: Native resolution 640x360 (16:9)
✓ Set resolution: 1920x1080 (16:9) using MJPG

Probing camera: /dev/video1
✗ Failed to open /dev/video1

Probing camera: /dev/video2
✗ Failed to open /dev/video2

Probing camera: /dev/video3
✗ Failed to open /dev/video3

Probing camera: /dev/video4


[ WARN:0@0.973] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name
[ WARN:0@0.973] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name
[ WARN:0@0.973] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name


Camera /dev/video4: Native resolution 640x480 (4:3)
✓ Set resolution: 640x480 (4:3) using MJPG

Probing camera: /dev/video5
✗ Failed to open /dev/video5

--- Detected Cameras ---
Camera #0:
  Name: OpenCV Camera @ /dev/video0
  Type: OpenCV
  Id: /dev/video0
  Backend api: V4L2
  Default stream profile:
    Format: MJPG
    Fourcc_code: 1196444237
    Width: 1920
    Height: 1080
    Fps: 30.0
    Aspect_ratio: 16:9 (1.778)
    Aspect_ratio_numeric: 1.7777777777777777
--------------------
Camera #1:
  Name: OpenCV Camera @ /dev/video4
  Type: OpenCV
  Id: /dev/video4
  Backend api: V4L2
  Default stream profile:
    Format: MJPG
    Fourcc_code: 1497715271
    Width: 640
    Height: 480
    Fps: 30.0
    Aspect_ratio: 4:3 (1.333)
    Aspect_ratio_numeric: 1.3333333333333333
--------------------
Initiate OpenCVCamera obejct with width 1920 x height 1080
DEBUG - 1920 x 1080


[ WARN:0@2.588] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name


Resetting camera resolution to MJPG
Initiate OpenCVCamera obejct with width 640 x height 480
DEBUG - 640 x 480


Resetting camera resolution to MJPG


ERROR:lerobot.find_cameras:Error reading from OpenCV camera /dev/video4: OpenCVCamera(/dev/video4) read failed (status=False).
[ WARN:0@15.269] global cap_v4l.cpp:1049 tryIoctl VIDEOIO(V4L2:/dev/video4): select() timeout.



Finalizing image saving...
Image capture finished. Images saved to /home/doog/repository/lerobot/notebooks/output


In [2]:
cv2.imread("/home/doog/repository/lerobot/notebooks/output/opencv__dev_video0.png")

array([[[113, 132, 147],
        [112, 131, 146],
        [111, 130, 145],
        ...,
        [127, 143, 150],
        [125, 141, 148],
        [124, 140, 147]],

       [[112, 131, 146],
        [111, 130, 145],
        [110, 129, 144],
        ...,
        [129, 145, 152],
        [127, 143, 150],
        [126, 142, 149]],

       [[112, 131, 146],
        [110, 129, 144],
        [109, 128, 143],
        ...,
        [131, 147, 154],
        [130, 146, 153],
        [129, 145, 152]],

       ...,

       [[ 97, 106, 110],
        [100, 109, 113],
        [103, 112, 116],
        ...,
        [ 33,  36,  44],
        [ 30,  33,  41],
        [ 28,  31,  39]],

       [[109, 116, 119],
        [109, 116, 119],
        [109, 116, 119],
        ...,
        [ 33,  36,  41],
        [ 31,  34,  39],
        [ 30,  33,  38]],

       [[112, 119, 122],
        [111, 118, 121],
        [109, 116, 119],
        ...,
        [ 32,  35,  40],
        [ 32,  35,  40],
        [ 32,  35,  40]]

In [ ]:
MAX_OPENCV_INDEX = 60

possible_paths = sorted(Path("/dev").glob("video*"), key=lambda p: p.name)
targets_to_scan = [str(p) for p in possible_paths]

found_cameras_info = []

for target in targets_to_scan:

    camera = cv2.VideoCapture(target, cv2.CAP_V4L2)
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    if camera.isOpened():

        default_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
        default_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
        default_fps = camera.get(cv2.CAP_PROP_FPS)
        default_format = camera.get(cv2.CAP_PROP_FORMAT)
        camera_info = {
            "name": f"OpenCV Camera @ {target}",
            "type": "OpenCV",
            "id": target,
            "backend_api": camera.getBackendName(),
            "default_stream_profile": {
                "format": default_format,
                "width": default_width,
                "height": default_height,
                "fps": default_fps,
            },
        }
        found_cameras_info.append(camera_info)
        camera.release()


display(found_cameras_info)


[]

In [22]:
import cv2
from pathlib import Path
import math

MAX_OPENCV_INDEX = 60

possible_paths = sorted(Path("/dev").glob("video*"), key=lambda p: p.name)
targets_to_scan = [str(p) for p in possible_paths]

found_cameras_info = []

# Comprehensive resolution lists by aspect ratio
RESOLUTIONS_16_9 = [
    (1920, 1080),  # Full HD
    (1280, 720),   # HD
    (1024, 576),   # 576p
    (854, 480),    # 480p wide
    (640, 360),    # 360p wide
]

RESOLUTIONS_4_3 = [
    (1600, 1200),  # UXGA
    (1280, 960),   # 960p
    (1024, 768),   # XGA
    (800, 600),    # SVGA
    (640, 480),    # VGA
]

# Some cameras support both, so we'll also try mixed ratios
RESOLUTIONS_OTHER = [
    (1920, 1440),  # 4:3 at high res
    (1440, 1080),  # 4:3 variant
    (960, 720),    # 4:3 variant
]

def get_aspect_ratio(width, height):
    """Calculate and classify aspect ratio"""
    ratio = width / height
    
    if abs(ratio - 16/9) < 0.1:
        return "16:9", ratio
    elif abs(ratio - 4/3) < 0.1:
        return "4:3", ratio
    else:
        return "other", ratio

def probe_camera_native_resolution(camera):
    """Try to determine the camera's native/default aspect ratio"""
    # Get the camera's default resolution without setting anything
    default_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
    default_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
    aspect_type, aspect_ratio = get_aspect_ratio(default_width, default_height)
    
    return aspect_type, default_width, default_height

def try_set_camera_resolution(camera, target_width, target_height, preferred_formats=['MJPG', 'YUYV']):
    """Try to set camera resolution with different pixel formats"""
    
    for fmt in preferred_formats:
        # Set pixel format
        if fmt == 'MJPG':
            fourcc = cv2.VideoWriter.fourcc('M','J','P','G')
        elif fmt == 'YUYV':
            fourcc = cv2.VideoWriter.fourcc('Y','U','Y','V')
        elif fmt == 'H264':
            fourcc = cv2.VideoWriter.fourcc('H','2','6','4')
        else:
            continue
            
        camera.set(cv2.CAP_PROP_FOURCC, fourcc)
        camera.set(cv2.CAP_PROP_FRAME_WIDTH, target_width)
        camera.set(cv2.CAP_PROP_FRAME_HEIGHT, target_height)
        
        # Verify the resolution was actually set
        actual_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        if actual_width == target_width and actual_height == target_height:
            return True, fmt
    
    return False, None

def find_best_resolution_for_camera(camera, target_device):
    """Find the best resolution for a specific camera based on its characteristics"""
    
    # First, probe the camera's native aspect ratio
    native_aspect, default_width, default_height = probe_camera_native_resolution(camera)
    print(f"Camera {target_device}: Native resolution {default_width}x{default_height} ({native_aspect})")
    
    # Choose resolution list based on native aspect ratio
    if native_aspect == "16:9":
        primary_resolutions = RESOLUTIONS_16_9
        fallback_resolutions = RESOLUTIONS_4_3 + RESOLUTIONS_OTHER
    elif native_aspect == "4:3":
        primary_resolutions = RESOLUTIONS_4_3
        fallback_resolutions = RESOLUTIONS_16_9 + RESOLUTIONS_OTHER
    else:
        primary_resolutions = RESOLUTIONS_OTHER + RESOLUTIONS_16_9
        fallback_resolutions = RESOLUTIONS_4_3
    
    # Try primary resolutions first (matching native aspect ratio)
    for width, height in primary_resolutions:
        success, format_used = try_set_camera_resolution(camera, width, height)
        if success:
            aspect_type, aspect_ratio = get_aspect_ratio(width, height)
            return width, height, format_used, aspect_type, aspect_ratio
    
    # If primary fails, try fallback resolutions
    print(f"Primary aspect ratio resolutions failed for {target_device}, trying fallback...")
    for width, height in fallback_resolutions:
        success, format_used = try_set_camera_resolution(camera, width, height)
        if success:
            aspect_type, aspect_ratio = get_aspect_ratio(width, height)
            return width, height, format_used, aspect_type, aspect_ratio
    
    # If everything fails, return current settings
    current_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
    current_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
    current_fourcc = int(camera.get(cv2.CAP_PROP_FOURCC))
    format_used = "".join([chr((current_fourcc >> 8 * i) & 0xFF) for i in range(4)])
    aspect_type, aspect_ratio = get_aspect_ratio(current_width, current_height)
    
    return current_width, current_height, format_used, aspect_type, aspect_ratio

for target in targets_to_scan:
    print(f"\nProbing camera: {target}")
    camera = cv2.VideoCapture(target, cv2.CAP_V4L2)
    
    if camera.isOpened():
        # Find the best resolution for this specific camera
        final_width, final_height, format_used, aspect_type, aspect_ratio = find_best_resolution_for_camera(camera, target)
        
        # Get other camera properties
        final_fps = camera.get(cv2.CAP_PROP_FPS)
        final_fourcc = int(camera.get(cv2.CAP_PROP_FOURCC))
        
        camera_info = {
            "name": f"OpenCV Camera @ {target}",
            "type": "OpenCV",
            "id": target,
            "backend_api": camera.getBackendName(),
            "default_stream_profile": {
                "format": format_used,
                "fourcc_code": final_fourcc,
                "width": final_width,
                "height": final_height,
                "fps": final_fps,
                "aspect_ratio": f"{aspect_type} ({aspect_ratio:.3f})",
                "aspect_ratio_numeric": aspect_ratio,
            },
        }
        found_cameras_info.append(camera_info)
        print(f"✓ Set resolution: {final_width}x{final_height} ({aspect_type}) using {format_used}")
        camera.release()
    else:
        print(f"✗ Failed to open {target}")

display(found_cameras_info)



Probing camera: /dev/video0
Camera /dev/video0: Native resolution 640x360 (16:9)
✓ Set resolution: 1920x1080 (16:9) using MJPG

Probing camera: /dev/video1
✗ Failed to open /dev/video1

Probing camera: /dev/video2
✗ Failed to open /dev/video2

Probing camera: /dev/video3
✗ Failed to open /dev/video3

Probing camera: /dev/video4
Camera /dev/video4: Native resolution 640x480 (4:3)


[ WARN:0@1714.322] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name
[ WARN:0@1714.322] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name
[ WARN:0@1714.322] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name


✓ Set resolution: 640x480 (4:3) using MJPG

Probing camera: /dev/video5
✗ Failed to open /dev/video5


[ WARN:0@1715.728] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name


[{'name': 'OpenCV Camera @ /dev/video0',
  'type': 'OpenCV',
  'id': '/dev/video0',
  'backend_api': 'V4L2',
  'default_stream_profile': {'format': 'MJPG',
   'fourcc_code': 1196444237,
   'width': 1920,
   'height': 1080,
   'fps': 30.0,
   'aspect_ratio': '16:9 (1.778)',
   'aspect_ratio_numeric': 1.7777777777777777}},
 {'name': 'OpenCV Camera @ /dev/video4',
  'type': 'OpenCV',
  'id': '/dev/video4',
  'backend_api': 'V4L2',
  'default_stream_profile': {'format': 'MJPG',
   'fourcc_code': 1497715271,
   'width': 640,
   'height': 480,
   'fps': 30.0,
   'aspect_ratio': '4:3 (1.333)',
   'aspect_ratio_numeric': 1.3333333333333333}}]

In [13]:
found_cameras_info

[{'name': 'OpenCV Camera @ /dev/video0',
  'type': 'OpenCV',
  'id': '/dev/video0',
  'backend_api': 'V4L2',
  'default_stream_profile': {'format': 0.0,
   'width': 640,
   'height': 360,
   'fps': 30.0}},
 {'name': 'OpenCV Camera @ /dev/video4',
  'type': 'OpenCV',
  'id': '/dev/video4',
  'backend_api': 'V4L2',
  'default_stream_profile': {'format': 0.0,
   'width': 640,
   'height': 480,
   'fps': 30.0}}]

In [16]:
from lerobot.cameras.configs import ColorMode
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig

OpenCVCameraConfig("0", color_mode=ColorMode.RGB)

OpenCVCameraConfig(fps=None, width=None, height=None, index_or_path='0', color_mode=<ColorMode.RGB: 'rgb'>, rotation=<Cv2Rotation.NO_ROTATION: 0>, warmup_s=1)